## Konfigurasi

In [8]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


### Struktur Folder

In [3]:
# ============================================================
# MIGRASI STRUKTUR MODELING (AMAN — TANPA HAPUS)
# ============================================================

import os
import shutil

THESIS_ROOT = '/content/drive/MyDrive/THESIS'

OLD_MODELING = f'{THESIS_ROOT}/DATASETS/MODELING'
OLD_MODELS = f'{THESIS_ROOT}/DATASETS/MODELS'
OLD_RESULTS = f'{THESIS_ROOT}/DATASETS/RESULTS'

NEW_MODELING_ROOT = f'{THESIS_ROOT}/MODELING'
NEW_TASK_DIR = f'{NEW_MODELING_ROOT}/COGNITIVE DISTORTION'

# ------------------------------------------------------------------
# 1. Buat struktur baru
# ------------------------------------------------------------------
NEW_FOLDERS = [
    f'{NEW_TASK_DIR}/configs/models',
    f'{NEW_TASK_DIR}/src/baselines',
    f'{NEW_TASK_DIR}/src/finetune',
    f'{NEW_TASK_DIR}/scripts',
    f'{NEW_TASK_DIR}/notebooks',
    f'{NEW_TASK_DIR}/logs',
    f'{NEW_TASK_DIR}/models',
    f'{NEW_TASK_DIR}/results/predictions',
    f'{NEW_TASK_DIR}/results/metrics',
    f'{NEW_TASK_DIR}/results/figures',
    f'{NEW_TASK_DIR}/results/tables',
]
for f in NEW_FOLDERS:
    os.makedirs(f, exist_ok=True)
print(f"Struktur baru dibuat: {len(NEW_FOLDERS)} folder")

# ------------------------------------------------------------------
# 2. Helper copy dengan logging
# ------------------------------------------------------------------
def copy_or_merge(src, dst, label=""):
    """Copy atau merge folder. Log file yang di-overwrite."""
    if not os.path.exists(src):
        return 0, 0
    n_new, n_over = 0, 0
    for root, dirs, files in os.walk(src):
        rel = os.path.relpath(root, src)
        target = os.path.join(dst, rel) if rel != "." else dst
        os.makedirs(target, exist_ok=True)
        for f in files:
            s = os.path.join(root, f)
            d = os.path.join(target, f)
            if os.path.exists(d):
                n_over += 1
                print(f"  overwrite: {rel}/{f}")
            else:
                n_new += 1
            shutil.copy2(s, d)
    print(f"{label}: {n_new} file baru, {n_over} file ditimpa")
    return n_new, n_over

# ------------------------------------------------------------------
# 3. Copy dari MODELING lama
# ------------------------------------------------------------------
print()
print("Copy dari MODELING lama:")
if os.path.exists(f'{OLD_MODELING}/COGNITIVE DISTORTION'):
    src_root = f'{OLD_MODELING}/COGNITIVE DISTORTION'
    for sub in ['configs', 'src', 'scripts', 'notebooks', 'logs']:
        s = f'{src_root}/{sub}'
        d = f'{NEW_TASK_DIR}/{sub}'
        if os.path.exists(s):
            copy_or_merge(s, d, label=sub)
else:
    print("  (tidak ada)")

# ------------------------------------------------------------------
# 4. Copy dari MODELS lama
# ------------------------------------------------------------------
print()
print("Copy dari MODELS lama:")
if os.path.exists(f'{OLD_MODELS}/COGNITIVE DISTORTION'):
    src_root = f'{OLD_MODELS}/COGNITIVE DISTORTION'
    d = f'{NEW_TASK_DIR}/models'
    copy_or_merge(src_root, d, label="models")
else:
    print("  (tidak ada)")

# ------------------------------------------------------------------
# 5. Copy dari RESULTS lama
# ------------------------------------------------------------------
print()
print("Copy dari RESULTS lama:")
if os.path.exists(f'{OLD_RESULTS}/COGNITIVE DISTORTION'):
    src_root = f'{OLD_RESULTS}/COGNITIVE DISTORTION'
    d = f'{NEW_TASK_DIR}/results'
    copy_or_merge(src_root, d, label="results")
else:
    print("  (tidak ada)")

# ------------------------------------------------------------------
# 6. Verifikasi — hitung file lama vs baru
# ------------------------------------------------------------------
print()
print("=" * 60)
print("VERIFIKASI COPY")
print("=" * 60)

def count_files(root):
    if not os.path.exists(root):
        return 0
    return sum(len(files) for _, _, files in os.walk(root))

pairs = [
    ("MODELING/COGNITIVE DISTORTION", f'{OLD_MODELING}/COGNITIVE DISTORTION', NEW_TASK_DIR),
    ("MODELS/COGNITIVE DISTORTION",   f'{OLD_MODELS}/COGNITIVE DISTORTION',   f'{NEW_TASK_DIR}/models'),
    ("RESULTS/COGNITIVE DISTORTION",  f'{OLD_RESULTS}/COGNITIVE DISTORTION',  f'{NEW_TASK_DIR}/results'),
]
all_match = True
for label, old, new in pairs:
    n_old = count_files(old)
    n_new = count_files(new)
    match = "OK" if n_new >= n_old else "MISMATCH"
    if n_new < n_old:
        all_match = False
    print(f"  {label}: lama={n_old}, baru={n_new}  [{match}]")

print()
if all_match:
    print("Semua file tersalin.")
    print()
    print("Setelah Anda yakin, jalankan cell terpisah untuk hapus folder lama:")
    print(f"  import shutil; shutil.rmtree('{OLD_MODELING}')")
    print(f"  import shutil; shutil.rmtree('{OLD_MODELS}')")
    print(f"  import shutil; shutil.rmtree('{OLD_RESULTS}')")
else:
    print("Ada file yang belum tersalin. JANGAN hapus folder lama dulu.")

Struktur baru dibuat: 11 folder

Copy dari MODELING lama:
  (tidak ada)

Copy dari MODELS lama:
  (tidak ada)

Copy dari RESULTS lama:
  (tidak ada)

VERIFIKASI COPY
  MODELING/COGNITIVE DISTORTION: lama=0, baru=21  [OK]
  MODELS/COGNITIVE DISTORTION: lama=0, baru=0  [OK]
  RESULTS/COGNITIVE DISTORTION: lama=0, baru=0  [OK]

Semua file tersalin.

Setelah Anda yakin, jalankan cell terpisah untuk hapus folder lama:
  import shutil; shutil.rmtree('/content/drive/MyDrive/THESIS/DATASETS/MODELING')
  import shutil; shutil.rmtree('/content/drive/MyDrive/THESIS/DATASETS/MODELS')
  import shutil; shutil.rmtree('/content/drive/MyDrive/THESIS/DATASETS/RESULTS')


### base.yaml & config_utils.py

## Cell — Setup Config Konsolidasi

**Fungsi:** Buat semua file config dari awal dalam satu cell:
- `base.yaml` — path, kolom, hyperparameter default
- `config_utils.py` — utility (versi lengkap dengan hash check + `get_fold_data`)
- 12 config model di `configs/models/`
- Folder checkpoint untuk 12 model

**Input:** —
**Output:** Semua file config + folder checkpoint

**Reset:** Folder `configs/models/` dihapus dulu, lalu dibuat ulang. Ini memastikan tidak ada config lama yang tertinggal.

**12 model:**
- Non-transformer: `tfidf_lr`, `tfidf_svm`, `svm_word2vec`
- Indonesia: `indobert_base_p1`, `indobert_15g`, `indoroberta_15g`, `indobertweet`, `nusabert`
- Multilingual: `mbert`, `xlmr`
- Kontribusi (placeholder): `indobert_dapt`, `indobert_dapt_tapt`

**Yang perlu diperiksa:**
- 8 fungsi inti ter-import: OK
- 12 model terdeteksi
- Config `indobert_dapt.yaml` dan `indobert_dapt_tapt.yaml` punya komentar "PLACEHOLDER"

In [14]:
# ============================================================
# CELL PERBAIKAN #1 — SETUP CONFIG KONSOLIDASI (12 MODEL)
# Menggantikan cell lama yang hanya buat 10 config.
# Aman dijalankan ulang: folder configs/models direset dulu.
# ============================================================

import os
import sys
import shutil
import importlib

NEW_TASK_DIR = '/content/drive/MyDrive/THESIS/MODELING/COGNITIVE DISTORTION'
SRC_DIR = f'{NEW_TASK_DIR}/src'
CONFIG_DIR = f'{NEW_TASK_DIR}/configs'
MODELS_CONFIG_DIR = f'{CONFIG_DIR}/models'
MODELS_DIR = f'{NEW_TASK_DIR}/models'

# ------------------------------------------------------------------
# 0. Reset folder config model (hapus lama, buat baru)
# ------------------------------------------------------------------
if os.path.exists(MODELS_CONFIG_DIR):
    shutil.rmtree(MODELS_CONFIG_DIR)
os.makedirs(MODELS_CONFIG_DIR, exist_ok=True)
os.makedirs(SRC_DIR, exist_ok=True)
os.makedirs(MODELS_DIR, exist_ok=True)
print("Folder config direset")

# ------------------------------------------------------------------
# 1. base.yaml
# ------------------------------------------------------------------
BASE_YAML_PATH = f'{CONFIG_DIR}/base.yaml'
base_yaml = '''paths:
  processed_dataset_dir: /content/drive/MyDrive/THESIS/DATASETS/PROCESSED DATASETS/COGNITIVE DISTORTION
  models_root: /content/drive/MyDrive/THESIS/MODELING/COGNITIVE DISTORTION/models
  results_root: /content/drive/MyDrive/THESIS/MODELING/COGNITIVE DISTORTION/results

data:
  text_column: "TEXT"
  label_column: "label_11"
  id_column: "sentence_id"
  group_column: "group_id"
  num_labels: 11
  max_seq_length: 128

training:
  seed: 42
  num_epochs: 5
  early_stopping_patience: 2
  metric_for_best_model: "macro_f1"
  greater_is_better: true
  class_weighted_loss: true
  fp16: true

eval:
  batch_size: 32
  save_predictions: true
'''
with open(BASE_YAML_PATH, 'w', encoding='utf-8') as f:
    f.write(base_yaml)
print("base.yaml dibuat")

# ------------------------------------------------------------------
# 2. config_utils.py (versi lengkap, dengan import json di atas)
# ------------------------------------------------------------------
CU_PATH = f'{SRC_DIR}/config_utils.py'

config_utils_code = '''"""Utility untuk load config, validasi data, dan set seed."""
import os
import sys
import random
import json
import copy
import yaml
import numpy as np


MODELING_ROOT = "/content/drive/MyDrive/THESIS/MODELING/COGNITIVE DISTORTION"
CONFIG_DIR = os.path.join(MODELING_ROOT, "configs")
MODELS_CONFIG_DIR = os.path.join(CONFIG_DIR, "models")


def _require(cfg, keys, context=""):
    cur = cfg
    path_so_far = []
    for k in keys:
        path_so_far.append(k)
        if not isinstance(cur, dict) or k not in cur:
            raise KeyError(
                f"Key '{'.'.join(path_so_far)}' tidak ditemukan"
                + (f" ({context})" if context else "")
            )
        cur = cur[k]
    return cur


def _deep_merge(base, override):
    result = copy.deepcopy(base)
    for key, value in override.items():
        if (key in result
                and isinstance(result[key], dict)
                and isinstance(value, dict)):
            result[key] = _deep_merge(result[key], value)
        else:
            result[key] = value
    return result


def load_config(model_name=None, config_path=None):
    if config_path is not None:
        with open(config_path, encoding="utf-8") as f:
            return yaml.safe_load(f)

    base_path = os.path.join(CONFIG_DIR, "base.yaml")
    if not os.path.exists(base_path):
        raise FileNotFoundError(f"base.yaml tidak ditemukan: {base_path}")
    with open(base_path, encoding="utf-8") as f:
        base = yaml.safe_load(f) or {}

    if model_name is None:
        return base

    model_path = os.path.join(MODELS_CONFIG_DIR, f"{model_name}.yaml")
    if not os.path.exists(model_path):
        raise FileNotFoundError(
            f"Config model tidak ditemukan: {model_path}. "
            f"Pilihan: {list_models()}"
        )
    with open(model_path, encoding="utf-8") as f:
        model_cfg = yaml.safe_load(f) or {}

    merged = _deep_merge(base, model_cfg)
    merged.setdefault("model_name", model_name)
    return merged


def list_models():
    if not os.path.isdir(MODELS_CONFIG_DIR):
        return []
    return sorted(
        f.replace(".yaml", "")
        for f in os.listdir(MODELS_CONFIG_DIR)
        if f.endswith(".yaml")
    )


def get_paths(model_name=None, cfg=None):
    if cfg is None:
        cfg = load_config()

    processed_dir = _require(cfg, ["paths", "processed_dataset_dir"], "base.yaml")
    models_root = _require(cfg, ["paths", "models_root"], "base.yaml")
    results_root = _require(cfg, ["paths", "results_root"], "base.yaml")

    paths = {
        "processed_dataset_dir": processed_dir,
        "dataset_csv": os.path.join(processed_dir, "dib_labeled.csv"),
        "folds_json": os.path.join(processed_dir, "folds.json"),
        "groups_json": os.path.join(processed_dir, "dib_groups.json"),
        "models_root": models_root,
        "results_root": results_root,
        "predictions_dir": os.path.join(results_root, "predictions"),
        "metrics_dir": os.path.join(results_root, "metrics"),
        "figures_dir": os.path.join(results_root, "figures"),
        "tables_dir": os.path.join(results_root, "tables"),
    }

    if model_name is not None:
        paths["model_dir"] = os.path.join(models_root, model_name)
        paths["model_predictions_dir"] = os.path.join(
            results_root, "predictions", model_name
        )
        paths["model_metrics_dir"] = os.path.join(
            results_root, "metrics", model_name
        )

    return paths


def ensure_dirs(*paths):
    for p in paths:
        if p:
            os.makedirs(p, exist_ok=True)


def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)
    try:
        import torch
        torch.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False
    except ImportError:
        pass


def _sha256_file(path):
    import hashlib
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(8192), b""):
            h.update(chunk)
    return h.hexdigest()


def validate_dataset(cfg=None, paths=None, verbose=True):
    import pandas as pd

    if paths is None:
        paths = get_paths(cfg=cfg)
    if cfg is None:
        cfg = load_config()

    dataset_csv = paths["dataset_csv"]
    if not os.path.exists(dataset_csv):
        raise FileNotFoundError(f"Dataset tidak ditemukan: {dataset_csv}")

    df = pd.read_csv(dataset_csv)

    id_col = cfg["data"]["id_column"]
    text_col = cfg["data"]["text_column"]
    label_col = cfg["data"]["label_column"]
    group_col = cfg["data"]["group_column"]
    required_cols = [id_col, text_col, label_col, group_col]

    missing = [c for c in required_cols if c not in df.columns]
    if missing:
        raise ValueError(
            f"Kolom wajib tidak ada: {missing}. "
            f"Kolom tersedia: {list(df.columns)}"
        )

    num_labels_cfg = cfg["data"]["num_labels"]
    labels_unique = sorted(df[label_col].dropna().unique().tolist())
    expected_labels = list(range(num_labels_cfg))
    label_ok = (labels_unique == expected_labels)

    nan_counts = {
        "id": int(df[id_col].isna().sum()),
        "text": int(df[text_col].isna().sum()),
        "label": int(df[label_col].isna().sum()),
        "group": int(df[group_col].isna().sum()),
    }
    total_nan = sum(nan_counts.values())

    n_dup_ids = int(df[id_col].duplicated().sum())
    ids_unique = (n_dup_ids == 0)

    label_dist = df[label_col].value_counts().sort_index().to_dict()
    n_rows = len(df)
    imbalance = (
        max(label_dist.values()) / min(label_dist.values())
        if label_dist else None
    )

    hash_status = "not_checked"
    hash_expected = None
    hash_actual = None
    folds_json = paths.get("folds_json")
    if folds_json and os.path.exists(folds_json):
        with open(folds_json, encoding="utf-8") as f:
            folds_meta = json.load(f).get("metadata", {})
        hash_expected = folds_meta.get("data_hash")
        hash_actual = _sha256_file(dataset_csv)
        if hash_expected and hash_expected == hash_actual:
            hash_status = "match"
        elif hash_expected:
            hash_status = "mismatch"
        else:
            hash_status = "no_hash_in_folds"

    summary = {
        "n_rows": n_rows,
        "n_labels_data": len(labels_unique),
        "n_labels_config": num_labels_cfg,
        "labels_unique": labels_unique,
        "labels_expected": expected_labels,
        "label_ok": label_ok,
        "nan_counts": nan_counts,
        "total_nan": total_nan,
        "ids_unique": ids_unique,
        "n_dup_ids": n_dup_ids,
        "label_distribution": label_dist,
        "class_imbalance_ratio": imbalance,
        "hash_status": hash_status,
        "hash_expected": hash_expected,
        "hash_actual": hash_actual,
    }

    if verbose:
        print("=" * 60)
        print("VALIDASI DATASET")
        print("=" * 60)
        print(f"File       : {dataset_csv}")
        print(f"Total baris: {n_rows}")
        print()
        print(f"Kolom wajib        : {required_cols}")
        print(f"Kolom tersedia     : {list(df.columns)}")
        print()
        print(f"Label unik di data       : {labels_unique}")
        print(f"Label diharapkan (config): {expected_labels}")
        print(f"Label cocok              : {'YA' if label_ok else 'TIDAK'}")
        print()
        print(f"sentence_id unik : {'YA' if ids_unique else 'TIDAK'} "
              f"(duplikat: {n_dup_ids})")
        print()
        print("NaN per kolom:")
        for k, v in nan_counts.items():
            print(f"  {k}: {v}")
        print()
        print(f"Distribusi label ({len(label_dist)} kelas):")
        for lbl, cnt in label_dist.items():
            pct = cnt / n_rows * 100
            print(f"  Label {lbl:>2}: {cnt:>5} ({pct:5.2f}%)")
        if imbalance:
            print()
            print(f"Rasio imbalance (max/min): {imbalance:.2f}")
        print()
        if hash_status == "match":
            print(f"Hash dataset vs folds.json: COCOK ({hash_actual[:16]}...)")
        elif hash_status == "mismatch":
            print(f"Hash dataset vs folds.json: BEDA")
            print(f"  diharapkan: {hash_expected[:16]}...")
            print(f"  aktual    : {hash_actual[:16]}...")
        elif hash_status == "no_hash_in_folds":
            print("Hash dataset: tidak ada di metadata folds.json")
        else:
            print("Hash dataset: folds.json tidak ada, dilewati")
        print("=" * 60)

    if not label_ok:
        raise ValueError(
            f"Label TIDAK cocok dengan config. "
            f"Data: {labels_unique}, Config: {expected_labels}."
        )
    if total_nan > 0:
        raise ValueError(f"Ada NaN di kolom wajib: {nan_counts}")
    if not ids_unique:
        raise ValueError(
            f"Ada {n_dup_ids} sentence_id duplikat. Periksa pipeline preprocessing."
        )

    return summary


def load_folds(paths=None, cfg=None, verbose=True):
    if paths is None:
        paths = get_paths(cfg=cfg)

    folds_json = paths["folds_json"]
    if not os.path.exists(folds_json):
        raise FileNotFoundError(f"folds.json tidak ditemukan: {folds_json}")

    with open(folds_json, encoding="utf-8") as f:
        data = json.load(f)

    if "folds" not in data or "metadata" not in data:
        raise ValueError("Skema folds.json tidak sesuai (folds/metadata).")

    required_meta = ["data_hash", "n_splits", "seed", "total_rows"]
    missing_meta = [k for k in required_meta if k not in data["metadata"]]
    if missing_meta:
        raise ValueError(f"Metadata folds.json kurang: {missing_meta}")

    for fold_name, fold in data["folds"].items():
        for split in ("train_ids", "val_ids", "test_ids"):
            if split not in fold:
                raise ValueError(f"{fold_name} tidak punya key '{split}'")

    dataset_csv = paths["dataset_csv"]
    if not os.path.exists(dataset_csv):
        raise FileNotFoundError(f"Dataset tidak ditemukan: {dataset_csv}")

    hash_actual = _sha256_file(dataset_csv)
    hash_expected = data["metadata"]["data_hash"]

    if hash_actual != hash_expected:
        raise ValueError(
            f"Hash dataset TIDAK cocok dengan folds.json. "
            f"Dataset sudah berubah sejak fold dibuat. "
            f"diharapkan: {hash_expected[:16]}..., aktual: {hash_actual[:16]}... "
            f"Jalankan ulang preprocessing build_folds."
        )

    for fold_name, fold in data["folds"].items():
        train = set(fold["train_ids"])
        val = set(fold["val_ids"])
        test = set(fold["test_ids"])
        if train & val:
            raise ValueError(f"{fold_name}: train dan val tidak kosong")
        if train & test:
            raise ValueError(f"{fold_name}: train dan test tidak kosong")
        if val & test:
            raise ValueError(f"{fold_name}: val dan test tidak kosong")

    if verbose:
        print("=" * 60)
        print("LOAD FOLDS")
        print("=" * 60)
        print(f"File      : {folds_json}")
        print(f"Data hash : {hash_expected[:16]}... (cocok dengan dataset)")
        print(f"N splits  : {data['metadata']['n_splits']}")
        print(f"Seed      : {data['metadata']['seed']}")
        print(f"Total rows: {data['metadata']['total_rows']}")
        print()
        for name, fold in data["folds"].items():
            print(f"{name}: train={len(fold['train_ids'])}, "
                  f"val={len(fold['val_ids'])}, test={len(fold['test_ids'])}")
        print("=" * 60)

    return data


def get_fold_data(df, folds_data, fold_name, cfg):
    if fold_name not in folds_data["folds"]:
        raise KeyError(
            f"Fold '{fold_name}' tidak ada. "
            f"Tersedia: {list(folds_data['folds'].keys())}"
        )

    fold = folds_data["folds"][fold_name]
    id_col = cfg["data"]["id_column"]

    train_df = df[df[id_col].isin(fold["train_ids"])].reset_index(drop=True)
    val_df = df[df[id_col].isin(fold["val_ids"])].reset_index(drop=True)
    test_df = df[df[id_col].isin(fold["test_ids"])].reset_index(drop=True)

    return train_df, val_df, test_df
'''

with open(CU_PATH, 'w', encoding='utf-8') as f:
    f.write(config_utils_code)
print(f"config_utils.py dibuat ({os.path.getsize(CU_PATH)} bytes)")

# ------------------------------------------------------------------
# 3. 12 config model
# ------------------------------------------------------------------
model_configs = {
    # Non-transformer
    "tfidf_lr": '''model_name: tfidf_lr
model_type: sklearn
model_class: LogisticRegression
random_state: 42
hyperparameters:
  C: 1.0
  max_iter: 1000
  class_weight: balanced
vectorizer:
  max_features: 50000
  ngram_range: [1, 2]
  min_df: 2
''',
    "tfidf_svm": '''model_name: tfidf_svm
model_type: sklearn
model_class: LinearSVC
random_state: 42
hyperparameters:
  C: 1.0
  max_iter: 2000
  class_weight: balanced
vectorizer:
  max_features: 50000
  ngram_range: [1, 2]
  min_df: 2
''',
    "svm_word2vec": '''model_name: svm_word2vec
model_type: sklearn
model_class: SVC
random_state: 42
hyperparameters:
  kernel: rbf
  C: 1.0
  gamma: scale
  class_weight: balanced
word2vec:
  vector_size: 100
  window: 5
  min_count: 2
  epochs: 20
''',
    # Transformer Indonesia
    "indobert_base_p1": '''model_name: indobert_base_p1
model_type: transformer
hf_model_id: indobenchmark/indobert-base-p1
training:
  batch_size: 16
  learning_rate: 2.0e-5
  weight_decay: 0.01
  warmup_ratio: 0.1
  num_epochs: 5
''',
    "indobert_15g": '''model_name: indobert_15g
model_type: transformer
hf_model_id: cahya/bert-base-indonesian-1.5G
training:
  batch_size: 16
  learning_rate: 2.0e-5
  weight_decay: 0.01
  warmup_ratio: 0.1
  num_epochs: 5
''',
    "indoroberta_15g": '''model_name: indoroberta_15g
model_type: transformer
hf_model_id: cahya/roberta-base-indonesian-1.5G
training:
  batch_size: 16
  learning_rate: 2.0e-5
  weight_decay: 0.01
  warmup_ratio: 0.1
  num_epochs: 5
''',
    "indobertweet": '''model_name: indobertweet
model_type: transformer
hf_model_id: indolem/indobertweet-base-uncased
training:
  batch_size: 16
  learning_rate: 2.0e-5
  weight_decay: 0.01
  warmup_ratio: 0.1
  num_epochs: 5
''',
    "nusabert": '''model_name: nusabert
model_type: transformer
hf_model_id: LazarusNLP/NusaBERT-base
training:
  batch_size: 16
  learning_rate: 2.0e-5
  weight_decay: 0.01
  warmup_ratio: 0.1
  num_epochs: 5
''',
    # Multilingual
    "mbert": '''model_name: mbert
model_type: transformer
hf_model_id: bert-base-multilingual-cased
training:
  batch_size: 16
  learning_rate: 2.0e-5
  weight_decay: 0.01
  warmup_ratio: 0.1
  num_epochs: 5
''',
    "xlmr": '''model_name: xlmr
model_type: transformer
hf_model_id: xlm-roberta-base
training:
  batch_size: 16
  learning_rate: 2.0e-5
  weight_decay: 0.01
  warmup_ratio: 0.1
  num_epochs: 5
''',
    # Kontribusi (placeholder)
    "indobert_dapt": '''model_name: indobert_dapt
# PLACEHOLDER - backbone belum final, tunggu hasil Fase B (Kategori B).
# hf_model_id SEMENTARA, akan diupdate setelah keputusan backbone.
model_type: transformer
hf_model_id: indobenchmark/indobert-base-p1
dapt_model_dir: /content/drive/MyDrive/THESIS/MODELING/COGNITIVE DISTORTION/models/indobert_dapt
training:
  batch_size: 16
  learning_rate: 2.0e-5
  weight_decay: 0.01
  warmup_ratio: 0.1
  num_epochs: 5
''',
    "indobert_dapt_tapt": '''model_name: indobert_dapt_tapt
# PLACEHOLDER - backbone belum final, tunggu hasil Fase B (Kategori B).
# hf_model_id SEMENTARA, akan diupdate setelah keputusan backbone.
model_type: transformer
hf_model_id: indobenchmark/indobert-base-p1
tapt_model_dir_template: /content/drive/MyDrive/THESIS/MODELING/COGNITIVE DISTORTION/models/indobert_dapt_tapt/fold_{fold}
training:
  batch_size: 16
  learning_rate: 2.0e-5
  weight_decay: 0.01
  warmup_ratio: 0.1
  num_epochs: 5
''',
}

print()
print("Config model:")
for name, content in model_configs.items():
    path = f'{MODELS_CONFIG_DIR}/{name}.yaml'
    with open(path, 'w', encoding='utf-8') as f:
        f.write(content)
    print(f"  {name}.yaml")

print(f"\nTotal: {len(model_configs)} config model")

# ------------------------------------------------------------------
# 4. Folder checkpoint untuk 12 model
# ------------------------------------------------------------------
for name in model_configs.keys():
    os.makedirs(f'{MODELS_DIR}/{name}', exist_ok=True)
print(f"Folder checkpoint: {len(model_configs)} model")

# ------------------------------------------------------------------
# 5. Verifikasi
# ------------------------------------------------------------------
if SRC_DIR not in sys.path:
    sys.path.insert(0, SRC_DIR)

import config_utils as cu
importlib.reload(cu)

required_funcs = [
    'load_config', 'get_paths', 'set_seed', 'list_models',
    'ensure_dirs', 'validate_dataset', 'load_folds', 'get_fold_data',
]
missing = [fn for fn in required_funcs if not hasattr(cu, fn)]
if missing:
    raise RuntimeError(f"Fungsi hilang: {missing}")

print()
print("=" * 60)
print("VERIFIKASI")
print("=" * 60)
print(f"Fungsi inti    : {len(required_funcs)} OK")
print(f"Model tersedia : {len(cu.list_models())}")
for m in cu.list_models():
    print(f"  {m}")
print("=" * 60)

Folder config direset
base.yaml dibuat
config_utils.py dibuat (12220 bytes)

Config model:
  tfidf_lr.yaml
  tfidf_svm.yaml
  svm_word2vec.yaml
  indobert_base_p1.yaml
  indobert_15g.yaml
  indoroberta_15g.yaml
  indobertweet.yaml
  nusabert.yaml
  mbert.yaml
  xlmr.yaml
  indobert_dapt.yaml
  indobert_dapt_tapt.yaml

Total: 12 config model
Folder checkpoint: 12 model

VERIFIKASI
Fungsi inti    : 8 OK
Model tersedia : 12
  indobert_15g
  indobert_base_p1
  indobert_dapt
  indobert_dapt_tapt
  indobertweet
  indoroberta_15g
  mbert
  nusabert
  svm_word2vec
  tfidf_lr
  tfidf_svm
  xlmr


### Validasi Dataset & Load Folds

## Cell — Verifikasi Config + Validasi Dataset + Load Folds

**Fungsi:**
- Reload `config_utils.py` setelah reset config folder
- Validasi `dib_labeled.csv` vs config (`num_labels`, kolom wajib, NaN, hash)
- Load `folds.json` + cek irisan split
- Test `get_fold_data()` untuk semua 5 fold

**Input:**
- `dib_labeled.csv` (dari PROCESSED DATASETS)
- `folds.json` (split 5-fold)

**Output:** Ringkasan validasi + folds ter-load + test get_fold_data

**Yang perlu diperiksa:**
- Hash dataset vs folds.json: `COCOK`
- Label cocok: 0–10 (11 kelas)
- NaN: 0 semua
- 5 fold ter-load tanpa error
- Total train+val+test per fold = 4.628

In [17]:
# ============================================================
# CELL A — PERBAIKI loader.py
# load_folds sekarang reuse cu.load_folds (validasi lengkap)
# ============================================================

import os

SRC_DIR = '/content/drive/MyDrive/THESIS/MODELING/COGNITIVE DISTORTION/src'
LOADER_PATH = f'{SRC_DIR}/loader.py'

loader_code = '''"""Load dataset dan folds untuk modeling."""
import os
import sys
import json
import pandas as pd

sys.path.insert(0, "/content/drive/MyDrive/THESIS/MODELING/COGNITIVE DISTORTION/src")
import config_utils as cu


def load_dataset(cfg, paths):
    return pd.read_csv(paths["dataset_csv"])


def load_folds(paths, verbose=True):
    """Reuse cu.load_folds: validasi hash + skema + irisan split."""
    return cu.load_folds(paths=paths, verbose=verbose)


def get_fold_data(df, folds_data, fold_name, cfg):
    """Filter DataFrame per fold. Pakai id_column dari cfg."""
    if fold_name not in folds_data["folds"]:
        raise KeyError(
            f"Fold '{fold_name}' tidak ada. "
            f"Tersedia: {list(folds_data['folds'].keys())}"
        )

    fold = folds_data["folds"][fold_name]
    id_col = cfg["data"]["id_column"]

    train_df = df[df[id_col].isin(fold["train_ids"])].reset_index(drop=True)
    val_df = df[df[id_col].isin(fold["val_ids"])].reset_index(drop=True)
    test_df = df[df[id_col].isin(fold["test_ids"])].reset_index(drop=True)
    return train_df, val_df, test_df


def get_texts_labels(df, cfg):
    text_col = cfg["data"]["text_column"]
    label_col = cfg["data"]["label_column"]
    return df[text_col].tolist(), df[label_col].tolist()
'''

with open(LOADER_PATH, 'w', encoding='utf-8') as f:
    f.write(loader_code)
print(f"loader.py diperbaiki ({os.path.getsize(LOADER_PATH)} bytes)")

# Verifikasi
import importlib
if SRC_DIR not in sys.path:
    sys.path.insert(0, SRC_DIR)

import loader
importlib.reload(loader)

# Test: load_folds dengan verbose + tanpa verbose
paths = cu.get_paths()
print("\nTest loader.load_folds(paths, verbose=False):")
fd = loader.load_folds(paths, verbose=False)
print(f"  OK — {len(fd['folds'])} fold")

print("\nTest loader.load_folds(paths, verbose=True):")
fd = loader.load_folds(paths, verbose=True)
print("  OK")

loader.py diperbaiki (1272 bytes)

Test loader.load_folds(paths, verbose=False):
  OK — 5 fold

Test loader.load_folds(paths, verbose=True):
LOAD FOLDS
File      : /content/drive/MyDrive/THESIS/DATASETS/PROCESSED DATASETS/COGNITIVE DISTORTION/folds.json
Data hash : ed154162554b68ba... (cocok dengan dataset)
N splits  : 5
Seed      : 42
Total rows: 4628

fold_0: train=3345, val=364, test=919
fold_1: train=3333, val=371, test=924
fold_2: train=3340, val=365, test=923
fold_3: train=3322, val=367, test=939
fold_4: train=3327, val=378, test=923
  OK


In [16]:
# ============================================================
# VERIFIKASI CONFIG + VALIDASI DATASET + LOAD FOLDS
# ============================================================

import os
import sys
import importlib
import pandas as pd

SRC_DIR = '/content/drive/MyDrive/THESIS/MODELING/COGNITIVE DISTORTION/src'
if SRC_DIR not in sys.path:
    sys.path.insert(0, SRC_DIR)

import config_utils as cu
importlib.reload(cu)

cu.set_seed(42)

paths = cu.get_paths()

# ------------------------------------------------------------------
# 1. Validasi dataset
# ------------------------------------------------------------------
summary = cu.validate_dataset(paths=paths)

# ------------------------------------------------------------------
# 2. Load folds
# ------------------------------------------------------------------
print()
folds_data = cu.load_folds(paths=paths)

# ------------------------------------------------------------------
# 3. Test get_fold_data untuk semua fold
# ------------------------------------------------------------------
print()
print("=" * 60)
print("TEST get_fold_data UNTUK SEMUA FOLD")
print("=" * 60)

df = pd.read_csv(paths["dataset_csv"])
cfg = cu.load_config()

for fold_name in sorted(folds_data["folds"].keys()):
    train_df, val_df, test_df = cu.get_fold_data(df, folds_data, fold_name, cfg)
    total = len(train_df) + len(val_df) + len(test_df)
    print(f"{fold_name}: train={len(train_df)}, val={len(val_df)}, "
          f"test={len(test_df)}, total={total}")
print("=" * 60)

VALIDASI DATASET
File       : /content/drive/MyDrive/THESIS/DATASETS/PROCESSED DATASETS/COGNITIVE DISTORTION/dib_labeled.csv
Total baris: 4628

Kolom wajib        : ['sentence_id', 'TEXT', 'label_11', 'group_id']
Kolom tersedia     : ['sentence_id', 'TEXT', 'DATA STATUS', 'FIRST ANNOTATOR', 'SECOND ANNOTATOR', 'TEXT_ORIGINAL', 'SPAN_START', 'SPAN_END', 'SPAN_TEXT', 'N_DOLLAR', 'IS_ANOMALY', 'HAS_VALID_SPAN', 'SPAN_TYPE', 'label_11', 'n_chars', 'group_id']

Label unik di data       : [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10]
Label diharapkan (config): [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10]
Label cocok              : YA

sentence_id unik : YA (duplikat: 0)

NaN per kolom:
  id: 0
  text: 0
  label: 0
  group: 0

Distribusi label (11 kelas):
  Label  0:  2219 (47.95%)
  Label  1:   445 ( 9.62%)
  Label  2:   388 ( 8.38%)
  Label  3:   371 ( 8.02%)
  Label  4:   239 ( 5.16%)
  Label  5:   188 ( 4.06%)
  Label  6:   283 ( 6.11%)
  Label  7:   214 ( 4.62%)
  Label  8:   158 ( 3.41%)
  Label  9:    85 (

## Baseline Sanity

## Cell — Majority-Class Baseline

**Fungsi:** Jalankan baseline paling sederhana — model yang selalu prediksi kelas mayoritas (label 0) di setiap fold.

**Tujuan:** Memberi batas bawah (lower bound). Kalau model lain tidak jauh lebih baik, ada masalah serius di data atau setup.

**Input:**
- `dib_labeled.csv`
- `folds.json`
- Modul: `loader.py`, `baselines/sklearn_baseline.py`

**Output:**
- `results/predictions/majority_class/fold_*.csv` — prediksi per fold
- `results/metrics/majority_class/fold_*.json` — metrik per fold
- `results/metrics/majority_class/summary.json` — ringkasan

**Metrik yang diharapkan:**
- Macro-F1: ~0.04 (sangat rendah, karena kelas 1–10 tidak pernah diprediksi)
- Weighted-F1: ~0.31 (hanya kelas 0 yang berkontribusi)
- Accuracy: ~0.48 (menipu — kelas 0 dominan 48%)
- Per-class F1:
  - Label 0: ~0.65
  - Label 1–10: 0.0

**Catatan:** Accuracy terlihat "lumayan" (0.48), tapi itu menipu. Macro-F1 = 0.04 menunjukkan model sebenarnya tidak berguna. Ini mengapa Macro-F1 dipilih sebagai metrik utama.

### Majority Class

In [6]:
# ============================================================
# BASELINE: MAJORITY-CLASS
# ============================================================

import os
import sys
import importlib

SRC_DIR = '/content/drive/MyDrive/THESIS/MODELING/COGNITIVE DISTORTION/src'
BASELINE_DIR = f'{SRC_DIR}/baselines'

for p in [SRC_DIR, BASELINE_DIR]:
    if p not in sys.path:
        sys.path.insert(0, p)

import config_utils as cu
import loader
import baselines.sklearn_baseline as sb
importlib.reload(cu)
importlib.reload(loader)
importlib.reload(sb)

cu.set_seed(42)

paths = cu.get_paths()
cfg = cu.load_config()
folds_data = loader.load_folds(paths)
df = loader.load_dataset(cfg, paths)

print("=" * 60)
print("JALANKAN: majority_class")
print("=" * 60)

summary = sb.run_majority_class(df, folds_data, cfg, paths)

print()
print("=" * 60)
print("RINGKASAN majority_class")
print("=" * 60)
print(f"Folds       : {summary['n_folds']}")
print(f"Macro-F1    : {summary['macro_f1_mean']:.4f} ± {summary['macro_f1_std']:.4f}")
print(f"Weighted-F1 : {summary['weighted_f1_mean']:.4f} ± {summary['weighted_f1_std']:.4f}")
print(f"Accuracy    : {summary['accuracy_mean']:.4f} ± {summary['accuracy_std']:.4f}")
print()
print("Per-class F1 (mean ± std):")
for lbl in range(11):
    f1 = summary['per_class_f1_mean'][str(lbl)]
    std = summary['per_class_f1_std'][str(lbl)]
    print(f"  Label {lbl:>2}: {f1:.4f} ± {std:.4f}")
print("=" * 60)

JALANKAN: majority_class

RINGKASAN majority_class
Folds       : 5
Macro-F1    : 0.0589 ± 0.0002
Weighted-F1 : 0.3108 ± 0.0026
Accuracy    : 0.4795 ± 0.0024

Per-class F1 (mean ± std):
  Label  0: 0.6482 ± 0.0022
  Label  1: 0.0000 ± 0.0000
  Label  2: 0.0000 ± 0.0000
  Label  3: 0.0000 ± 0.0000
  Label  4: 0.0000 ± 0.0000
  Label  5: 0.0000 ± 0.0000
  Label  6: 0.0000 ± 0.0000
  Label  7: 0.0000 ± 0.0000
  Label  8: 0.0000 ± 0.0000
  Label  9: 0.0000 ± 0.0000
  Label 10: 0.0000 ± 0.0000


### TF-IDF + Logistic Regression

## Cell — TF-IDF + Logistic Regression Baseline

**Fungsi:** Baseline leksikal dengan TF-IDF (1–2 gram) + Logistic Regression.

**Tujuan:** Menguji apakah fitur leksikal sederhana sudah cukup untuk klasifikasi 11 kelas. Kalau TF-IDF + LR sudah bagus, transformer (IndoBERT) tidak memberi nilai tambah signifikan.

**Input:**
- `dib_labeled.csv`
- `folds.json`
- Config: `configs/models/tfidf_lr.yaml`

**Output:**
- `results/predictions/tfidf_lr/fold_*.csv`
- `results/metrics/tfidf_lr/summary.json`

**Hyperparameter (dari YAML):**
- TF-IDF: `max_features=50000`, `ngram_range=(1,2)`, `min_df=2`
- LR: `C=1.0`, `max_iter=1000`, `class_weight=balanced`, `random_state=42`

**Metrik yang diharapkan:**
- Macro-F1: ~0.4–0.6 (perkiraan, jauh di atas majority-class 0.06)
- Accuracy: ~0.5–0.7
- Per-class F1 kelas minoritas mulai terisi (tidak lagi 0)

**Waktu eksekusi:** Menit (CPU), bukan jam.

**Catatan:** `class_weight=balanced` penting karena imbalance 58×. Tanpa ini, model cenderung prediksi kelas mayoritas saja.

In [7]:
# ============================================================
# JALANKAN BASELINE: TF-IDF + LOGISTIC REGRESSION
# ============================================================

import os
import sys
import importlib

SRC_DIR = '/content/drive/MyDrive/THESIS/MODELING/COGNITIVE DISTORTION/src'
BASELINE_DIR = f'{SRC_DIR}/baselines'

for p in [SRC_DIR, BASELINE_DIR]:
    if p not in sys.path:
        sys.path.insert(0, p)

import config_utils as cu
import loader
import baselines.sklearn_baseline as sb
importlib.reload(cu)
importlib.reload(loader)
importlib.reload(sb)

cu.set_seed(42)

paths = cu.get_paths()
cfg = cu.load_config()
folds_data = loader.load_folds(paths)
df = loader.load_dataset(cfg, paths)

print("=" * 60)
print("JALANKAN: tfidf_lr")
print("=" * 60)

summary = sb.run_tfidf_lr(df, folds_data, cfg, paths)

print()
print("=" * 60)
print("RINGKASAN tfidf_lr")
print("=" * 60)
print(f"Folds       : {summary['n_folds']}")
print(f"Macro-F1    : {summary['macro_f1_mean']:.4f} ± {summary['macro_f1_std']:.4f}")
print(f"Weighted-F1 : {summary['weighted_f1_mean']:.4f} ± {summary['weighted_f1_std']:.4f}")
print(f"Accuracy    : {summary['accuracy_mean']:.4f} ± {summary['accuracy_std']:.4f}")
print()
print("Per-class F1 (mean ± std):")
for lbl in range(11):
    f1 = summary['per_class_f1_mean'][str(lbl)]
    std = summary['per_class_f1_std'][str(lbl)]
    print(f"  Label {lbl:>2}: {f1:.4f} ± {std:.4f}")
print("=" * 60)

JALANKAN: tfidf_lr

RINGKASAN tfidf_lr
Folds       : 5
Macro-F1    : 0.5022 ± 0.0094
Weighted-F1 : 0.5776 ± 0.0152
Accuracy    : 0.5764 ± 0.0136

Per-class F1 (mean ± std):
  Label  0: 0.5757 ± 0.0234
  Label  1: 0.5613 ± 0.0254
  Label  2: 0.5583 ± 0.0317
  Label  3: 0.7929 ± 0.0421
  Label  4: 0.6280 ± 0.0611
  Label  5: 0.6234 ± 0.0223
  Label  6: 0.6098 ± 0.0490
  Label  7: 0.4697 ± 0.0368
  Label  8: 0.4626 ± 0.0307
  Label  9: 0.2427 ± 0.0854
  Label 10: 0.0000 ± 0.0000


### TF-IDF + Linear SVM

## Cell — TF-IDF + Linear SVM Baseline

**Fungsi:** Baseline leksikal dengan TF-IDF (1–2 gram) + Linear SVM.

**Tujuan:** Bandingkan classifier berbeda pada fitur yang sama. Linear SVM sering lebih baik di text classification karena berbasis margin, bukan probabilitas.

**Perbedaan dengan Cell TF-IDF + LR:**
- TF-IDF + LR: probabilistic (softmax)
- TF-IDF + SVM: margin-based
- Hasil biasanya mirip, kadang SVM sedikit lebih baik, kadang sebaliknya

**Input:**
- `dib_labeled.csv`
- `folds.json`
- Config: `configs/models/tfidf_svm.yaml`

**Output:**
- `results/predictions/tfidf_svm/fold_*.csv`
- `results/metrics/tfidf_svm/summary.json`

**Hyperparameter (dari YAML):**
- TF-IDF: `max_features=50000`, `ngram_range=(1,2)`, `min_df=2`
- SVM: `C=1.0`, `max_iter=2000`, `class_weight=balanced`, `random_state=42`

**Metrik yang diharapkan:**
- Macro-F1: mirip atau sedikit berbeda dari TF-IDF + LR
- Accuracy: sama
- Per-class F1: mirip

**Waktu eksekusi:** Menit (CPU).

**Catatan:** Linear SVM tidak punya `predict_proba` (kecuali pakai `CalibratedClassifierCV`). Untuk XAI nanti, kita pakai model transformer yang punya probabilitas eksplisit.

In [9]:
# ============================================================
# JALANKAN BASELINE: TF-IDF + LINEAR SVM
# ============================================================

import os
import sys
import importlib

SRC_DIR = '/content/drive/MyDrive/THESIS/MODELING/COGNITIVE DISTORTION/src'
BASELINE_DIR = f'{SRC_DIR}/baselines'

for p in [SRC_DIR, BASELINE_DIR]:
    if p not in sys.path:
        sys.path.insert(0, p)

import config_utils as cu
import loader
import baselines.sklearn_baseline as sb
importlib.reload(cu)
importlib.reload(loader)
importlib.reload(sb)

cu.set_seed(42)

paths = cu.get_paths()
cfg = cu.load_config()
folds_data = loader.load_folds(paths)
df = loader.load_dataset(cfg, paths)

print("=" * 60)
print("JALANKAN: tfidf_svm")
print("=" * 60)

summary = sb.run_tfidf_svm(df, folds_data, cfg, paths)

print()
print("=" * 60)
print("RINGKASAN tfidf_svm")
print("=" * 60)
print(f"Folds       : {summary['n_folds']}")
print(f"Macro-F1    : {summary['macro_f1_mean']:.4f} ± {summary['macro_f1_std']:.4f}")
print(f"Weighted-F1 : {summary['weighted_f1_mean']:.4f} ± {summary['weighted_f1_std']:.4f}")
print(f"Accuracy    : {summary['accuracy_mean']:.4f} ± {summary['accuracy_std']:.4f}")
print()
print("Per-class F1 (mean ± std):")
for lbl in range(11):
    f1 = summary['per_class_f1_mean'][str(lbl)]
    std = summary['per_class_f1_std'][str(lbl)]
    print(f"  Label {lbl:>2}: {f1:.4f} ± {std:.4f}")
print("=" * 60)

JALANKAN: tfidf_svm

RINGKASAN tfidf_svm
Folds       : 5
Macro-F1    : 0.5106 ± 0.0164
Weighted-F1 : 0.6342 ± 0.0155
Accuracy    : 0.6334 ± 0.0172

Per-class F1 (mean ± std):
  Label  0: 0.6888 ± 0.0177
  Label  1: 0.6019 ± 0.0067
  Label  2: 0.5643 ± 0.0291
  Label  3: 0.7893 ± 0.0269
  Label  4: 0.6597 ± 0.0364
  Label  5: 0.5988 ± 0.0536
  Label  6: 0.6046 ± 0.0401
  Label  7: 0.4371 ± 0.0589
  Label  8: 0.4550 ± 0.0336
  Label  9: 0.2170 ± 0.1396
  Label 10: 0.0000 ± 0.0000


## Cell — Tambah + Jalankan Baseline: SVM (RBF) + Word2Vec

**Fungsi:**
1. Tambahkan fungsi `run_svm_word2vec` ke `sklearn_baseline.py` (kalau belum ada)
2. Jalankan baseline DIB: SVM (RBF) + Word2Vec

**Tujuan:** Baseline dari paper DIB. Membandingkan metode DIB dengan metode kita nanti.

**Input:**
- `dib_labeled.csv`
- `folds.json`
- Config: `configs/models/svm_word2vec.yaml`
- Modul: `sklearn_baseline.py`

**Output:**
- `results/predictions/svm_word2vec/fold_*.csv`
- `results/metrics/svm_word2vec/summary.json`

**Metode:**
- Latih Word2Vec pada **teks train fold** (bukan pre-trained publik) untuk fairness
- Konversi teks → vektor: rata-rata vektor kata
- SVM RBF: `C=1.0`, `gamma='scale'`, `class_weight='balanced'`

**Hyperparameter (dari YAML):**
- Word2Vec: `vector_size=100`, `window=5`, `min_count=2`, `epochs=20`
- SVM: `kernel='rbf'`, `C=1.0`, `gamma='scale'`

**Metrik yang diharapkan:**
- Macro-F1: ~0.3–0.5 (perkiraan)
- Weighted-F1: ~0.4–0.6
- Accuracy: ~0.4–0.6

**Waktu eksekusi:** Beberapa menit (CPU) — Word2Vec + SVM per fold.

**Catatan:**
- Word2Vec dilatih ulang untuk setiap fold → tidak ada kebocoran data
- Hasil ini yang menjadi baseline DIB resmi di tesis

In [11]:
# ============================================================
# INSTALL GENSIM + JALANKAN SVM (RBF) + WORD2VEC
# ============================================================

import sys
import subprocess

# Install gensim
print("Install gensim...")
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'gensim'], check=True)
print("Gensim installed")

# ------------------------------------------------------------------
# Reload + jalankan
# ------------------------------------------------------------------
import os
import importlib

SRC_DIR = '/content/drive/MyDrive/THESIS/MODELING/COGNITIVE DISTORTION/src'
for p in [SRC_DIR, f'{SRC_DIR}/baselines']:
    if p not in sys.path:
        sys.path.insert(0, p)

import config_utils as cu
import loader
import baselines.sklearn_baseline as sb
importlib.reload(cu)
importlib.reload(loader)
importlib.reload(sb)

cu.set_seed(42)

paths = cu.get_paths()
cfg = cu.load_config()
folds_data = loader.load_folds(paths)
df = loader.load_dataset(cfg, paths)

print()
print("=" * 60)
print("JALANKAN: svm_word2vec")
print("=" * 60)

summary = sb.run_svm_word2vec(df, folds_data, cfg, paths)

print()
print("=" * 60)
print("RINGKASAN svm_word2vec")
print("=" * 60)
print(f"Folds       : {summary['n_folds']}")
print(f"Macro-F1    : {summary['macro_f1_mean']:.4f} ± {summary['macro_f1_std']:.4f}")
print(f"Weighted-F1 : {summary['weighted_f1_mean']:.4f} ± {summary['weighted_f1_std']:.4f}")
print(f"Accuracy    : {summary['accuracy_mean']:.4f} ± {summary['accuracy_std']:.4f}")
print()
print("Per-class F1 (mean ± std):")
for lbl in range(11):
    f1 = summary['per_class_f1_mean'][str(lbl)]
    std = summary['per_class_f1_std'][str(lbl)]
    print(f"  Label {lbl:>2}: {f1:.4f} ± {std:.4f}")
print("=" * 60)

Install gensim...
Gensim installed

JALANKAN: svm_word2vec

RINGKASAN svm_word2vec
Folds       : 5
Macro-F1    : 0.3628 ± 0.0117
Weighted-F1 : 0.3806 ± 0.0165
Accuracy    : 0.3818 ± 0.0144

Per-class F1 (mean ± std):
  Label  0: 0.3250 ± 0.0267
  Label  1: 0.4666 ± 0.0301
  Label  2: 0.4162 ± 0.0391
  Label  3: 0.5834 ± 0.0552
  Label  4: 0.4673 ± 0.0306
  Label  5: 0.4431 ± 0.0237
  Label  6: 0.4494 ± 0.0210
  Label  7: 0.3308 ± 0.0388
  Label  8: 0.2701 ± 0.0241
  Label  9: 0.1847 ± 0.0796
  Label 10: 0.0540 ± 0.0247


## Cell — Ringkasan Semua Baseline Sanity

**Fungsi:** Kumpulkan hasil semua baseline sanity dalam satu tabel perbandingan.

**Input:** `results/metrics/<model>/summary.json` untuk 4 model:
- `majority_class`
- `tfidf_lr`
- `tfidf_svm`
- `svm_word2vec`

**Output:**
- Tabel ringkasan (Macro-F1, Weighted-F1, Accuracy)
- Tabel per-class F1
- File `results/metrics/baseline_sanity_summary.json`

**Yang perlu diperiksa:**
- `majority_class` — Macro-F1 sangat rendah (~0.06), jadi batas bawah
- `tfidf_lr` / `tfidf_svm` — Macro-F1 ~0.4–0.6
- `svm_word2vec` — baseline DIB, patokan literatur
- Kelas minoritas (label 9, 10) — apakah F1 > 0?

**Interpretasi:**
- Kalau TF-IDF sudah sangat bagus (>0.7), transformer mungkin tidak memberi nilai tambah
- Kalau SVM+Word2Vec jauh lebih rendah dari TF-IDF, ada masalah di representasi vektor
- Semua model transformer (IndoBERT dst.) nanti harus **jauh di atas** baseline ini

**Waktu eksekusi:** Detik (hanya baca file JSON).

In [12]:
# ============================================================
# RINGKASAN SEMUA BASELINE SANITY
# ============================================================

import os
import sys
import json
import importlib

SRC_DIR = '/content/drive/MyDrive/THESIS/MODELING/COGNITIVE DISTORTION/src'
if SRC_DIR not in sys.path:
    sys.path.insert(0, SRC_DIR)

import config_utils as cu
importlib.reload(cu)

paths = cu.get_paths()
metrics_dir = paths["metrics_dir"]

# Daftar baseline sanity
baselines = ["majority_class", "tfidf_lr", "tfidf_svm", "svm_word2vec"]

print("=" * 78)
print("RINGKASAN BASELINE SANITY (5-fold)")
print("=" * 78)
print(f"{'Model':<20} {'Macro-F1':>12} {'Weighted-F1':>14} {'Accuracy':>12}")
print("-" * 78)

summary_all = {}
for name in baselines:
    summary_path = os.path.join(metrics_dir, name, "summary.json")
    if not os.path.exists(summary_path):
        print(f"{name:<20} {'(belum dijalankan)':>40}")
        continue
    with open(summary_path, encoding='utf-8') as f:
        s = json.load(f)
    summary_all[name] = s
    macro = f"{s['macro_f1_mean']:.4f}±{s['macro_f1_std']:.4f}"
    weighted = f"{s['weighted_f1_mean']:.4f}±{s['weighted_f1_std']:.4f}"
    acc = f"{s['accuracy_mean']:.4f}±{s['accuracy_std']:.4f}"
    print(f"{name:<20} {macro:>12} {weighted:>14} {acc:>12}")

print("=" * 78)

# Per-class F1 comparison
if summary_all:
    print()
    print("PER-CLASS F1 (mean)")
    print("-" * 78)
    header = f"{'Label':<8}"
    for name in summary_all.keys():
        header += f" {name[:12]:>13}"
    print(header)
    print("-" * 78)
    for lbl in range(11):
        row = f"{lbl:<8}"
        for name, s in summary_all.items():
            f1 = s['per_class_f1_mean'][str(lbl)]
            row += f" {f1:>13.4f}"
        print(row)
    print("=" * 78)

# Simpan ringkasan ke JSON
if summary_all:
    out_path = os.path.join(metrics_dir, "baseline_sanity_summary.json")
    with open(out_path, 'w', encoding='utf-8') as f:
        json.dump(summary_all, f, indent=2)
    print(f"\nRingkasan disimpan: {out_path}")

RINGKASAN BASELINE SANITY (5-fold)
Model                    Macro-F1    Weighted-F1     Accuracy
------------------------------------------------------------------------------
majority_class       0.0589±0.0002  0.3108±0.0026 0.4795±0.0024
tfidf_lr             0.5022±0.0094  0.5776±0.0152 0.5764±0.0136
tfidf_svm            0.5106±0.0164  0.6342±0.0155 0.6334±0.0172
svm_word2vec         0.3628±0.0117  0.3806±0.0165 0.3818±0.0144

PER-CLASS F1 (mean)
------------------------------------------------------------------------------
Label     majority_cla      tfidf_lr     tfidf_svm  svm_word2vec
------------------------------------------------------------------------------
0               0.6482        0.5757        0.6888        0.3250
1               0.0000        0.5613        0.6019        0.4666
2               0.0000        0.5583        0.5643        0.4162
3               0.0000        0.7929        0.7893        0.5834
4               0.0000        0.6280        0.6597        0.4673

### Transformer

In [18]:
# ============================================================
# CELL B — PERBAIKI trainer.py
# class_weighted_loss flag sekarang benar-benar dipakai
# ============================================================

import os
import sys
import importlib

SRC_DIR = '/content/drive/MyDrive/THESIS/MODELING/COGNITIVE DISTORTION/src'
FINETUNE_DIR = f'{SRC_DIR}/finetune'
TRAINER_PATH = f'{FINETUNE_DIR}/trainer.py'

os.makedirs(FINETUNE_DIR, exist_ok=True)

trainer_code = '''"""Fine-tuning transformer untuk klasifikasi 11 kelas."""
import os
import sys
import json
import numpy as np
import pandas as pd
import torch
import torch.nn as nn

from torch.utils.data import Dataset, DataLoader
from transformers import (
    AutoTokenizer, AutoModelForSequenceClassification,
    get_linear_schedule_with_warmup,
)
from sklearn.metrics import f1_score

sys.path.insert(0, "/content/drive/MyDrive/THESIS/MODELING/COGNITIVE DISTORTION/src")
import config_utils as cu


class TextDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len=128):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        enc = self.tokenizer(
            str(self.texts[idx]),
            truncation=True,
            padding="max_length",
            max_length=self.max_len,
            return_tensors="pt",
        )
        return {
            "input_ids": enc["input_ids"].squeeze(0),
            "attention_mask": enc["attention_mask"].squeeze(0),
            "label": torch.tensor(self.labels[idx], dtype=torch.long),
        }


def _compute_class_weights(labels, num_labels):
    counts = np.bincount(labels, minlength=num_labels).astype(float)
    counts[counts == 0] = 1.0
    weights = len(labels) / (num_labels * counts)
    return torch.tensor(weights, dtype=torch.float)


def train_one_fold(
    model_name, hf_model_id,
    train_df, val_df, test_df,
    cfg, paths,
    num_labels=11, max_len=128, batch_size=16, learning_rate=2e-5,
    weight_decay=0.01, warmup_ratio=0.1, num_epochs=5, patience=2,
    fp16=True, device=None, fold_name="fold_0",
):
    if device is None:
        device = "cuda" if torch.cuda.is_available() else "cpu"

    id_col = cfg["data"]["id_column"]
    text_col = cfg["data"]["text_column"]
    label_col = cfg["data"]["label_column"]

    train_texts = train_df[text_col].tolist()
    train_labels = train_df[label_col].tolist()
    val_texts = val_df[text_col].tolist()
    val_labels = val_df[label_col].tolist()
    test_texts = test_df[text_col].tolist()
    test_labels = test_df[label_col].tolist()

    tokenizer = AutoTokenizer.from_pretrained(hf_model_id)
    model = AutoModelForSequenceClassification.from_pretrained(
        hf_model_id, num_labels=num_labels
    ).to(device)

    train_ds = TextDataset(train_texts, train_labels, tokenizer, max_len)
    val_ds = TextDataset(val_texts, val_labels, tokenizer, max_len)
    test_ds = TextDataset(test_texts, test_labels, tokenizer, max_len)

    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True)
    val_loader = DataLoader(val_ds, batch_size=batch_size)
    test_loader = DataLoader(test_ds, batch_size=batch_size)

    # Class-weighted loss: baca flag dari cfg
    use_class_weight = cfg["training"].get("class_weighted_loss", True)
    if use_class_weight:
        class_weights = _compute_class_weights(train_labels, num_labels).to(device)
        loss_fn = nn.CrossEntropyLoss(weight=class_weights)
    else:
        loss_fn = nn.CrossEntropyLoss()

    optimizer = torch.optim.AdamW(
        model.parameters(), lr=learning_rate, weight_decay=weight_decay
    )
    total_steps = len(train_loader) * num_epochs
    warmup_steps = int(total_steps * warmup_ratio)
    scheduler = get_linear_schedule_with_warmup(
        optimizer, num_warmup_steps=warmup_steps,
        num_training_steps=total_steps,
    )
    scaler = torch.cuda.amp.GradScaler() if fp16 and device == "cuda" else None

    def _eval(loader):
        model.eval()
        preds, probs_all, labels_all = [], [], []
        with torch.no_grad():
            for batch in loader:
                input_ids = batch["input_ids"].to(device)
                attn = batch["attention_mask"].to(device)
                labels = batch["label"].to(device)
                outputs = model(input_ids=input_ids, attention_mask=attn)
                logits = outputs.logits
                probs = torch.softmax(logits, dim=-1)
                preds.extend(torch.argmax(logits, dim=-1).cpu().numpy().tolist())
                probs_all.extend(probs.cpu().numpy().tolist())
                labels_all.extend(labels.cpu().numpy().tolist())
        macro_f1 = f1_score(labels_all, preds, average="macro", zero_division=0)
        return preds, probs_all, labels_all, macro_f1

    best_val_f1 = -1.0
    best_state = None
    patience_counter = 0
    history = []

    for epoch in range(num_epochs):
        model.train()
        total_loss = 0.0
        for batch in train_loader:
            optimizer.zero_grad()
            input_ids = batch["input_ids"].to(device)
            attn = batch["attention_mask"].to(device)
            labels = batch["label"].to(device)
            if scaler is not None:
                with torch.cuda.amp.autocast():
                    outputs = model(input_ids=input_ids, attention_mask=attn)
                    loss = loss_fn(outputs.logits, labels)
                scaler.scale(loss).backward()
                scaler.step(optimizer)
                scaler.update()
            else:
                outputs = model(input_ids=input_ids, attention_mask=attn)
                loss = loss_fn(outputs.logits, labels)
                loss.backward()
                optimizer.step()
            scheduler.step()
            total_loss += loss.item()

        avg_loss = total_loss / len(train_loader)
        _, _, _, val_f1 = _eval(val_loader)
        history.append({"epoch": epoch + 1, "train_loss": avg_loss, "val_macro_f1": val_f1})
        print(f"  Epoch {epoch+1}/{num_epochs} | loss={avg_loss:.4f} | val_macro_f1={val_f1:.4f}")

        if val_f1 > best_val_f1:
            best_val_f1 = val_f1
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
            patience_counter = 0
        else:
            patience_counter += 1
            if patience_counter >= patience:
                print(f"  Early stop di epoch {epoch+1}")
                break

    if best_state is not None:
        model.load_state_dict(best_state)
    model.to(device)

    _, _, _, val_f1 = _eval(val_loader)
    test_preds, test_probs, test_labels_eval, _ = _eval(test_loader)

    from metrics import compute_metrics
    val_metrics = {"macro_f1": val_f1}
    test_metrics = compute_metrics(test_labels_eval, test_preds, num_labels)
    test_metrics["history"] = history

    return val_metrics, test_metrics, test_preds, test_probs, test_labels_eval


def save_predictions_and_metrics(model_name, fold_name, test_df, test_preds,
                                 test_probs, test_metrics, cfg, paths):
    id_col = cfg["data"]["id_column"]
    pred_dir = os.path.join(paths["results_root"], "predictions", model_name)
    met_dir = os.path.join(paths["results_root"], "metrics", model_name)
    cu.ensure_dirs(pred_dir, met_dir)

    df_pred = pd.DataFrame({
        "sentence_id": test_df[id_col].tolist(),
        "y_true": test_df[cfg["data"]["label_column"]].tolist(),
        "y_pred": test_preds,
    })
    for i in range(cfg["data"]["num_labels"]):
        df_pred[f"prob_{i}"] = [p[i] for p in test_probs]
    df_pred.to_csv(os.path.join(pred_dir, f"{fold_name}.csv"), index=False)

    with open(os.path.join(met_dir, f"{fold_name}.json"), "w") as f:
        json.dump(test_metrics, f, indent=2)


def finalize(model_name, fold_metrics, paths, num_labels):
    from metrics import average_metrics
    met_dir = os.path.join(paths["results_root"], "metrics", model_name)
    cu.ensure_dirs(met_dir)
    summary = average_metrics(fold_metrics, num_labels)
    with open(os.path.join(met_dir, "summary.json"), "w") as f:
        json.dump(summary, f, indent=2)
    return summary
'''

with open(TRAINER_PATH, 'w', encoding='utf-8') as f:
    f.write(trainer_code)
print(f"trainer.py diperbaiki ({os.path.getsize(TRAINER_PATH)} bytes)")

# Verifikasi import
for p in [SRC_DIR, FINETUNE_DIR]:
    if p not in sys.path:
        sys.path.insert(0, p)

import finetune.trainer as tr
importlib.reload(tr)

print("trainer.py: OK")
print(f"Fungsi tersedia: {[f for f in dir(tr) if not f.startswith('_') and callable(getattr(tr, f))]}")

# Cek flag class_weighted_loss dibaca
cfg = cu.load_config() if 'cu' in dir() else None
if cfg is None:
    import config_utils as cu
    cfg = cu.load_config()
print(f"\nclass_weighted_loss di config: {cfg['training']['class_weighted_loss']}")

trainer.py diperbaiki (7813 bytes)
trainer.py: OK
Fungsi tersedia: ['AutoModelForSequenceClassification', 'AutoTokenizer', 'DataLoader', 'Dataset', 'TextDataset', 'f1_score', 'finalize', 'get_linear_schedule_with_warmup', 'save_predictions_and_metrics', 'train_one_fold']

class_weighted_loss di config: True


### Smoke Test

In [19]:
# ============================================================
# CELL C — SMOKE TEST: 1 MODEL × 1 FOLD
# Verifikasi pipeline + estimasi waktu sebelum 35 run
# ============================================================

import os
import sys
import time
import importlib

SRC_DIR = '/content/drive/MyDrive/THESIS/MODELING/COGNITIVE DISTORTION/src'
for p in [SRC_DIR, f'{SRC_DIR}/finetune']:
    if p not in sys.path:
        sys.path.insert(0, p)

import torch

if not torch.cuda.is_available():
    raise RuntimeError(
        "GPU tidak aktif. Runtime → Change runtime type → T4 GPU → Save."
    )

import config_utils as cu
import loader
import finetune.trainer as tr
importlib.reload(cu)
importlib.reload(loader)
importlib.reload(tr)

cu.set_seed(42)
paths = cu.get_paths()
cfg = cu.load_config()
folds_data = loader.load_folds(paths, verbose=False)
df = loader.load_dataset(cfg, paths)

model_name = "indobert_base_p1"
fold_name = "fold_0"

# Cek apakah sudah ada (resume)
met_path = os.path.join(paths["results_root"], "metrics", model_name, f"{fold_name}.json")
if os.path.exists(met_path):
    print(f"SKIP: {model_name} {fold_name} sudah ada di:")
    print(f"  {met_path}")
    print("Hapus file tersebut dulu kalau ingin ulang smoke test.")
else:
    model_cfg = cu.load_config(model_name)
    train_df, val_df, test_df = loader.get_fold_data(df, folds_data, fold_name, cfg)

    print("=" * 60)
    print(f"SMOKE TEST: {model_name} / {fold_name}")
    print("=" * 60)
    print(f"HF model            : {model_cfg['hf_model_id']}")
    print(f"Train / Val / Test  : {len(train_df)} / {len(val_df)} / {len(test_df)}")
    print(f"Device              : {torch.cuda.get_device_name(0)}")
    print(f"Batch size          : {model_cfg['training']['batch_size']}")
    print(f"Learning rate       : {model_cfg['training']['learning_rate']}")
    print(f"Num epochs          : {model_cfg['training']['num_epochs']}")
    print(f"Max seq length      : {cfg['data']['max_seq_length']}")
    print(f"Class weighted loss : {cfg['training']['class_weighted_loss']}")
    print(f"FP16                : {cfg['training']['fp16']}")
    print("=" * 60)
    print()

    t_start = time.time()

    val_metrics, test_metrics, test_preds, test_probs, test_labels = tr.train_one_fold(
        model_name=model_name,
        hf_model_id=model_cfg["hf_model_id"],
        train_df=train_df,
        val_df=val_df,
        test_df=test_df,
        cfg=cfg,
        paths=paths,
        num_labels=cfg["data"]["num_labels"],
        max_len=cfg["data"]["max_seq_length"],
        batch_size=model_cfg["training"]["batch_size"],
        learning_rate=model_cfg["training"]["learning_rate"],
        weight_decay=model_cfg["training"]["weight_decay"],
        warmup_ratio=model_cfg["training"]["warmup_ratio"],
        num_epochs=model_cfg["training"]["num_epochs"],
        patience=cfg["training"]["early_stopping_patience"],
        fp16=cfg["training"]["fp16"],
        fold_name=fold_name,
    )

    tr.save_predictions_and_metrics(
        model_name=model_name,
        fold_name=fold_name,
        test_df=test_df,
        test_preds=test_preds,
        test_probs=test_probs,
        test_metrics=test_metrics,
        cfg=cfg,
        paths=paths,
    )

    t_elapsed = time.time() - t_start

    print()
    print("=" * 60)
    print("SMOKE TEST BERHASIL")
    print("=" * 60)
    print(f"Val Macro-F1    : {val_metrics['macro_f1']:.4f}")
    print(f"Test Macro-F1   : {test_metrics['macro_f1']:.4f}")
    print(f"Test Weighted-F1: {test_metrics['weighted_f1']:.4f}")
    print(f"Test Accuracy   : {test_metrics['accuracy']:.4f}")
    print()
    print("Per-class F1 (test):")
    for lbl in range(11):
        f1 = test_metrics['per_class'][str(lbl)]['f1']
        print(f"  Label {lbl:>2}: {f1:.4f}")
    print()
    print(f"Waktu 1 fold    : {t_elapsed/60:.1f} menit ({t_elapsed:.0f} detik)")
    print(f"Estimasi 35 run : {t_elapsed * 35 / 3600:.1f} jam ({t_elapsed * 35 / 60:.0f} menit)")
    print("=" * 60)

RuntimeError: GPU tidak aktif. Runtime → Change runtime type → T4 GPU → Save.

## Cell — Fine-tuning Semua Model Transformer × 5 Fold

**Fungsi:** Jalankan fine-tuning 7 model transformer pada 5 fold:
- `indobert_base_p1`
- `indobert_15g`
- `indoroberta_15g`
- `indobertweet`
- `nusabert`
- `mbert`
- `xlmr`

**Fitur penting:**
- **Resume:** Kalau fold sudah pernah selesai, skip. Jalankan ulang cell kalau Colab timeout.
- **Continue on error:** Kalau satu model gagal (OOM, dll), lanjut ke model berikutnya.
- **Estimasi progress:** Setelah setiap model selesai, print estimasi sisa waktu.
- **Simpan per fold:** Aman kalau crash di tengah.

**Input:**
- `dib_labeled.csv`, `folds.json`
- 7 config model
- Modul `finetune/trainer.py`

**Output per model:**
- `results/predictions/<model>/fold_*.csv` (prediksi + probabilitas 11 kelas)
- `results/metrics/<model>/fold_*.json`
- `results/metrics/<model>/summary.json` (rata-rata 5 fold)
- `results/metrics/transformer_summary.json` (semua model)

**Waktu eksekusi (perkiraan):**
| Device | Per fold | 7 model × 5 fold |
|---|---|---|
| T4 | 15–25 menit | 8–14 jam |
| A100 | 5–10 menit | 3–6 jam |
| CPU | 2–4 jam | (tidak realistis) |

**Yang perlu diperiksa:**
- Macro-F1 transformer **jauh di atas** TF-IDF (0.51) dan SVM+Word2Vec (0.36)
- Per-class F1 kelas minoritas (label 9, 10) mulai > 0
- Konsistensi antar fold (std kecil)

**Kalau Colab timeout di tengah:**
- Jalankan ulang cell yang sama
- Fold yang sudah selesai akan di-skip
- Lanjut dari fold/model berikutnya

In [ ]:
# ============================================================
# FINE-TUNING SEMUA MODEL TRANSFORMER × 5 FOLD
# Fitur: resume (skip fold yang sudah selesai), continue on error
# ============================================================

import os
import sys
import time
import json
import importlib
import traceback

SRC_DIR = '/content/drive/MyDrive/THESIS/MODELING/COGNITIVE DISTORTION/src'
for p in [SRC_DIR, f'{SRC_DIR}/finetune']:
    if p not in sys.path:
        sys.path.insert(0, p)

import config_utils as cu
import loader
import finetune.trainer as tr
importlib.reload(cu)
importlib.reload(loader)
importlib.reload(tr)

cu.set_seed(42)

paths = cu.get_paths()
cfg_full = cu.load_config()
folds_data = loader.load_folds(paths, verbose=False)
df = loader.load_dataset(cfg_full, paths)
fold_names = sorted(folds_data["folds"].keys())

# Model transformer yang akan dijalankan
# (DAPT/TAPT belum ada checkpoint, jangan dimasukkan)
TRANSFORMER_MODELS = [
    "indobert_base_p1",
    "indobert_15g",
    "indoroberta_15g",
    "indobertweet",
    "nusabert",
    "mbert",
    "xlmr",
]

print("=" * 70)
print("FINE-TUNING SEMUA MODEL TRANSFORMER × 5 FOLD")
print("=" * 70)
print(f"Models   : {len(TRANSFORMER_MODELS)}")
print(f"Folds    : {len(fold_names)}")
print(f"Total run: {len(TRANSFORMER_MODELS) * len(fold_names)}")
print(f"Device   : {'cuda' if __import__('torch').cuda.is_available() else 'cpu'}")
print("=" * 70)
print()

# Tracking
results = {}
errors = []
t_global_start = time.time()

for model_name in TRANSFORMER_MODELS:
    print()
    print("#" * 70)
    print(f"# MODEL: {model_name}")
    print("#" * 70)

    try:
        model_cfg = cu.load_config(model_name)
        hf_model_id = model_cfg["hf_model_id"]
        tr_cfg = model_cfg["training"]
    except Exception as e:
        print(f"  [SKIP] Gagal load config: {e}")
        errors.append({"model": model_name, "stage": "config", "error": str(e)})
        continue

    print(f"HF model      : {hf_model_id}")
    print(f"Batch size    : {tr_cfg['batch_size']}")
    print(f"Learning rate : {tr_cfg['learning_rate']}")
    print()

    model_fold_metrics = []
    t_model_start = time.time()

    for fold_name in fold_names:
        # Cek apakah fold sudah selesai (resume)
        met_path = os.path.join(
            paths["results_root"], "metrics", model_name, f"{fold_name}.json"
        )
        if os.path.exists(met_path):
            print(f"  {fold_name}: SKIP (sudah ada)")
            with open(met_path) as f:
                model_fold_metrics.append(json.load(f))
            continue

        print(f"  {fold_name}: mulai...")
        t_fold_start = time.time()

        try:
            train_df, val_df, test_df = loader.get_fold_data(
                df, folds_data, fold_name, cfg_full
            )

            val_metrics, test_metrics, test_preds, test_probs, test_labels = \
                tr.train_one_fold(
                    model_name=model_name,
                    hf_model_id=hf_model_id,
                    train_df=train_df,
                    val_df=val_df,
                    test_df=test_df,
                    cfg=cfg_full,
                    paths=paths,
                    num_labels=cfg_full["data"]["num_labels"],
                    max_len=cfg_full["data"]["max_seq_length"],
                    batch_size=tr_cfg["batch_size"],
                    learning_rate=tr_cfg["learning_rate"],
                    weight_decay=tr_cfg["weight_decay"],
                    warmup_ratio=tr_cfg["warmup_ratio"],
                    num_epochs=tr_cfg["num_epochs"],
                    patience=cfg_full["training"]["early_stopping_patience"],
                    fp16=cfg_full["training"]["fp16"],
                    fold_name=fold_name,
                )

            tr.save_predictions_and_metrics(
                model_name=model_name,
                fold_name=fold_name,
                test_df=test_df,
                test_preds=test_preds,
                test_probs=test_probs,
                test_metrics=test_metrics,
                cfg=cfg_full,
                paths=paths,
            )

            model_fold_metrics.append(test_metrics)
            t_fold = time.time() - t_fold_start
            print(f"    test_macro_f1={test_metrics['macro_f1']:.4f} | "
                  f"waktu={t_fold/60:.1f} menit")

        except Exception as e:
            t_fold = time.time() - t_fold_start
            print(f"    [ERROR] {type(e).__name__}: {e}")
            print(f"    Waktu sebelum error: {t_fold/60:.1f} menit")
            errors.append({
                "model": model_name,
                "fold": fold_name,
                "error": f"{type(e).__name__}: {e}",
                "traceback": traceback.format_exc()[:1000],
            })
            continue

    # Finalize kalau ada minimal 1 fold sukses
    if model_fold_metrics:
        try:
            summary = tr.finalize(
                model_name, model_fold_metrics, paths,
                cfg_full["data"]["num_labels"]
            )
            results[model_name] = summary
            t_model = time.time() - t_model_start
            print()
            print(f"  RINGKASAN {model_name}:")
            print(f"    Macro-F1    : {summary['macro_f1_mean']:.4f} "
                  f"± {summary['macro_f1_std']:.4f}")
            print(f"    Weighted-F1 : {summary['weighted_f1_mean']:.4f} "
                  f"± {summary['weighted_f1_std']:.4f}")
            print(f"    Accuracy    : {summary['accuracy_mean']:.4f} "
                  f"± {summary['accuracy_std']:.4f}")
            print(f"    Waktu model : {t_model/60:.1f} menit")
        except Exception as e:
            print(f"  [ERROR] finalize: {e}")
            errors.append({"model": model_name, "stage": "finalize", "error": str(e)})

    # Estimasi sisa
    n_done = len(results)
    t_elapsed = time.time() - t_global_start
    if n_done > 0:
        t_per_model = t_elapsed / n_done
        t_remaining = t_per_model * (len(TRANSFORMER_MODELS) - n_done)
        print(f"  Progress: {n_done}/{len(TRANSFORMER_MODELS)} model | "
              f"estimasi sisa: {t_remaining/3600:.1f} jam")

# ==================================================================
# RINGKASAN AKHIR
# ==================================================================
print()
print("=" * 70)
print("RINGKASAN AKHIR — SEMUA MODEL TRANSFORMER")
print("=" * 70)
print(f"{'Model':<22} {'Macro-F1':>14} {'Weighted-F1':>16} {'Accuracy':>14}")
print("-" * 70)
for model_name in TRANSFORMER_MODELS:
    if model_name not in results:
        print(f"{model_name:<22} {'(gagal/tidak selesai)':>46}")
        continue
    s = results[model_name]
    macro = f"{s['macro_f1_mean']:.4f}±{s['macro_f1_std']:.4f}"
    weighted = f"{s['weighted_f1_mean']:.4f}±{s['weighted_f1_std']:.4f}"
    acc = f"{s['accuracy_mean']:.4f}±{s['accuracy_std']:.4f}"
    print(f"{model_name:<22} {macro:>14} {weighted:>16} {acc:>14}")
print("=" * 70)

t_total = time.time() - t_global_start
print(f"\nTotal waktu: {t_total/3600:.2f} jam ({t_total/60:.0f} menit)")

if errors:
    print()
    print("=" * 70)
    print(f"ERROR ({len(errors)})")
    print("=" * 70)
    for e in errors:
        print(f"  {e.get('model', '?')} / {e.get('fold', e.get('stage', '?'))}: "
              f"{e['error']}")

# Simpan hasil
if results:
    out_path = os.path.join(paths["metrics_dir"], "transformer_summary.json")
    with open(out_path, "w") as f:
        json.dump(results, f, indent=2)
    print(f"\nRingkasan disimpan: {out_path}")